In [0]:
%sql
-- checking data types of columns 

DESCRIBE TABLE databricks_ukwest.ashu_etl.ashu_student_perform_bronze;

col_name,data_type,comment
student_id,string,null
gender,string,null
study_time_hours,string,null
attendance_percent,string,null
sleep_hours,string,null
parental_education,string,null
internet_access,string,null
extracurricular_activities,string,null
part_time_job,string,null
previous_grade,string,null


### transform column data type 


In [0]:
%sql
CREATE OR REPLACE TABLE
    databricks_ukwest.ashu_etl.ashu_student_perform_clean_silver
USING DELTA
AS

WITH cleaned_data AS (
    SELECT
        TRY_CAST(student_id AS INT) AS student_id,

        INITCAP(TRIM(gender)) AS gender,

        TRY_CAST(study_time_hours AS DOUBLE)
            AS study_time_hours,

        TRY_CAST(attendance_percent AS DOUBLE)
            AS attendance_percent,

        TRY_CAST(sleep_hours AS DOUBLE)
            AS sleep_hours,

        NULLIF(TRIM(parental_education), '')
            AS parental_education,

        INITCAP(LOWER(TRIM(internet_access)))
            AS internet_access,

        INITCAP(LOWER(TRIM(extracurricular_activities)))
            AS extracurricular_activities,

        INITCAP(LOWER(TRIM(part_time_job)))
            AS part_time_job,

        TRY_CAST(previous_grade AS DOUBLE)
            AS previous_grade,

        TRY_CAST(final_exam_score AS DOUBLE)
            AS final_exam_score,

        UPPER(TRIM(final_grade)) AS final_grade,

        bronze_ingested_at

    FROM  databricks_ukwest.ashu_etl.ashu_student_perform_bronze
)

SELECT DISTINCT
    *,
    current_timestamp() AS silver_processed_at

FROM cleaned_data

WHERE student_id IS NOT NULL
  AND final_exam_score BETWEEN 0 AND 100
  AND attendance_percent BETWEEN 0 AND 100;

num_affected_rows,num_inserted_rows


### rejected record for auditing purpose 

In [0]:
%sql
CREATE OR REPLACE TABLE
    databricks_ukwest.ashu_etl.ashu_student_performance_rejected_table_new
USING DELTA
AS

WITH cleaned_data AS (
    SELECT
        *,
        TRY_CAST(student_id AS INT) AS parsed_student_id,
        TRY_CAST(final_exam_score AS DOUBLE) AS parsed_exam_score,
        TRY_CAST(attendance_percent AS DOUBLE) AS parsed_attendance
    FROM databricks_ukwest.ashu_etl.ashu_student_perform_clean_silver
)

SELECT
    *,
    CASE
        WHEN parsed_student_id IS NULL
            THEN 'Invalid student_id'
        WHEN parsed_exam_score IS NULL
            THEN 'Invalid or missing exam score'
        WHEN parsed_exam_score NOT BETWEEN 0 AND 100
            THEN 'Exam score outside 0-100'
        WHEN parsed_attendance IS NULL
            THEN 'Invalid or missing attendance'
        WHEN parsed_attendance NOT BETWEEN 0 AND 100
            THEN 'Attendance outside 0-100'
    END AS rejection_reason,

    current_timestamp() AS rejected_at

FROM cleaned_data

WHERE parsed_student_id IS NULL
   OR parsed_exam_score IS NULL
   OR parsed_exam_score NOT BETWEEN 0 AND 100
   OR parsed_attendance IS NULL
   OR parsed_attendance NOT BETWEEN 0 AND 100;

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from databricks_ukwest.ashu_etl.ashu_student_perform_clean_silver
LIMIT 10;

student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade,bronze_ingested_at,silver_processed_at
23,Female,3.7,81.8,7.8,High School,No,No,No,74.8,73.4,C,2026-11-27T11:02:11.655Z,2026-11-27T11:18:37.515Z
33,Female,5.3,100.0,7.3,Masters,Yes,Yes,No,67.7,100.0,A,2026-11-27T11:02:11.655Z,2026-11-27T11:18:37.515Z
42,Male,4.5,74.0,6.7,High School,Yes,No,No,54.1,79.0,C,2026-11-27T11:02:11.655Z,2026-11-27T11:18:37.515Z
59,Male,1.5,89.0,6.2,Masters,Yes,No,No,59.4,74.5,C,2026-11-27T11:02:11.655Z,2026-11-27T11:18:37.515Z
80,Female,3.2,90.9,7.9,High School,Yes,No,No,72.4,87.4,B,2026-11-27T11:02:11.655Z,2026-11-27T11:18:37.515Z
81,Male,4.4,100.0,5.9,High School,Yes,No,No,78.2,97.6,A,2026-11-27T11:02:11.655Z,2026-11-27T11:18:37.515Z
98,Female,3.8,81.0,5.9,High School,No,No,No,87.5,88.5,B,2026-11-27T11:02:11.655Z,2026-11-27T11:18:37.515Z
114,Female,1.4,90.9,7.2,Bachelors,Yes,No,No,62.0,74.6,C,2026-11-27T11:02:11.655Z,2026-11-27T11:18:37.515Z
132,Male,2.0,91.3,4.1,null,Yes,Yes,No,62.4,75.7,C,2026-11-27T11:02:11.655Z,2026-11-27T11:18:37.515Z
136,Male,4.3,90.8,10.0,Bachelors,Yes,Yes,No,58.9,94.5,A,2026-11-27T11:02:11.655Z,2026-11-27T11:18:37.515Z
